# Several GPUs benchmark: DDP, FSDP2 and the layers spread, on consumer GPUs

<small>Copyright Alejandro Martínez Corriá and the Thinkube contributors. SPDX-License-Identifier: Apache-2.0</small>

This notebook trains LoRA and QLoRA with meepsquared on the GPUs of one machine and compares with one GPU. Two questions, each with the modes that answer it: for a model that fits one GPU (Llama 3.1 8B in BF16; Qwen3.8-27B in 4-bit), what a second GPU buys with DDP; for a model whose weights do not fit one GPU (Qwen3-14B in BF16: 29.5 GB on a 24 GB card), which of FSDP2 and the layer split train it, and how fast.

| Mode | What it is |
|---|---|
| `one` | one GPU, the reference: `torchrun --nproc_per_node 1` |
| `ddp` | one copy of the model per GPU, the gradients averaged (DistributedDataParallel) |
| `fsdp` | the model sharded across the GPUs (FSDP2); BF16 LoRA loads on the CPU first, QLoRA on each GPU, where bitsandbytes quantizes |
| `split` | the layers spread over the GPUs, one process, one GPU busy at a time: for a model that does not fit one GPU |

LoRA rank 16 on the seven projections, padding-free, no_robots conversations, the same batch per GPU in every mode. Each run is one call of `multi_gpu.py`, next to this notebook, and appends one line to `results/multi_gpu.jsonl`: the tokens per second of all GPUs together, each GPU's peak memory, the loss.

## Settings

`MODEL_SOURCE` says where the models come from: `"huggingface"` downloads them from the Hub (Llama 3.1 is gated: accept its licence and set `HF_TOKEN`); `"thinkube"` reads them from the cluster's model mirror on [Thinkube](https://thinkube.org), copied once into the cluster's storage, so no run downloads anything and a gated model needs no token. `RUNS` lists the runs: model, method, mode, GPUs, examples per GPU per step.

A notebook job can override any of these with its parameters. On Thinkube, a job on a node that shares its GPUs takes every GPU of the node by asking for all of them.

In [ ]:
PARAMETERS = globals().get("PARAMETERS", {})

MODEL_SOURCE = PARAMETERS.get("MODEL_SOURCE", "huggingface")      # "thinkube" or "huggingface"
RUNS = PARAMETERS.get("RUNS", [
    # model, method, mode, GPUs, examples per GPU per step
    # fits one GPU: what a second one buys (DDP)
    ["meta-llama/Llama-3.1-8B-Instruct", "lora", "one", 1, 8],
    ["meta-llama/Llama-3.1-8B-Instruct", "lora", "ddp", 2, 8],
    ["Qwen/Qwen3.8-27B", "qlora", "one", 1, 8],
    ["Qwen/Qwen3.8-27B", "qlora", "ddp", 2, 8],
    # does not fit one GPU (29.5 GB of BF16 weights on a 24 GB card): FSDP2 and the split are the ways to train it
    ["Qwen/Qwen3-14B", "lora", "fsdp", 2, 8],
    ["Qwen/Qwen3-14B", "lora", "split", 2, 8],
])
STEPS = PARAMETERS.get("STEPS", 20)
WARMUP = PARAMETERS.get("WARMUP", 5)
# The benchmarks folder of meepsquared: this notebook's folder, unless a job runs a copy elsewhere
BENCHMARKS = PARAMETERS.get("BENCHMARKS")

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import torch

BENCHMARKS = Path(BENCHMARKS) if BENCHMARKS else Path.cwd()
if not (BENCHMARKS / "multi_gpu.py").is_file():
    raise FileNotFoundError(f"multi_gpu.py is not in {BENCHMARKS}: run this notebook from the benchmarks folder of meepsquared, "
                            "or set BENCHMARKS to that folder")
RESULTS = BENCHMARKS / "results" / "multi_gpu.jsonl"

sys.path.insert(0, str(BENCHMARKS))
from common import environment

print(json.dumps(environment(), indent=1))
print(f"{torch.cuda.device_count()} GPUs:", ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())))

## Where the models come from

On Thinkube, `thinkube_models.mirrored_model_path` gives the folder of each mirrored model, and stops with an error naming the model if one is not mirrored, before any training starts.

In [ ]:
models = sorted({run[0] for run in RUNS})
if MODEL_SOURCE == "thinkube":
    import thinkube_models as tkm

    for model in models:
        print(f"{model}: {tkm.mirrored_model_path(model)}")
else:
    print("Models are downloaded from the Hugging Face Hub")

## Run

Each run prints its steps as it goes. A run that fails stops the notebook with its own error. The Triton kernels a run compiles are kept in a cache folder next to the results; the first steps of a run are not measured.

In [ ]:
CACHE = BENCHMARKS / ".triton-cache"


def run(model, method, mode, gpus, batch):
    if gpus > torch.cuda.device_count():
        raise RuntimeError(f"{mode} on {gpus} GPUs: this server has {torch.cuda.device_count()}")

    script = [str(BENCHMARKS / "multi_gpu.py"), "--mode", "ddp" if mode == "one" else mode, "--model", model,
              "--source", MODEL_SOURCE, "--batch", str(batch), "--steps", str(STEPS), "--warmup", str(WARMUP),
              "--results", str(RESULTS)] + (["--qlora"] if method == "qlora" else [])
    if mode == "split":
        command = [sys.executable, "-u", *script]
    else:
        command = [sys.executable, "-m", "torch.distributed.run", "--nproc_per_node", str(gpus), *script]
    print(f"=== {model} {method} {mode} on {gpus} GPU(s)", flush=True)
    process = subprocess.Popen(command, cwd=BENCHMARKS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, "TOKENIZERS_PARALLELISM": "false", "TRITON_CACHE_DIR": str(CACHE)})
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError(f"{model} {method} {mode} failed with exit code {process.returncode}")


for model, method, mode, gpus, batch in RUNS:
    run(model, method, mode, gpus, batch)

## Results

Tokens per second count the tokens trained on, without padding, over all GPUs, from step `WARMUP` to the last. The peak memory is each GPU's. The loss of the last five steps shows whether the modes trained alike.

In [ ]:
import pandas as pd

rows = [json.loads(line) for line in RESULTS.read_text().splitlines()]
table = pd.DataFrame([r for r in rows if r.get("source") == MODEL_SOURCE])
table["mode"] = table.apply(lambda r: "one" if r["mode"] == "ddp" and r["processes"] == 1 else r["mode"], axis=1)
table = table.drop_duplicates(["model", "method", "mode", "batch_per_gpu"], keep="last")
display(table[["model", "method", "mode", "processes", "batch_per_gpu", "tokens_per_s", "s_per_step", "peak_gib", "loss_last5"]])